### Adriana Ximena GUzman Rojas
### Master IA

**Situación 1 — Outlier en `dias_ausencia_anio`**

Hay un trabajador con 187 días de ausencia en el año. Un año laboral en Colombia tiene aproximadamente 240 días hábiles.

Tarea: detecten ese outlier con IQR, decidan qué hacer con él y justifiquen su decisión en un comentario en el notebook.

```python
# Detecten el outlier con IQR
# Decidan: ¿eliminar la fila, corregir el valor, o conservarlo?
# Escriban la justificación como comentario en esta celda
```

**Situación 2 — Decisión de imputación para `nivel_estres`**

Volvamos a `nivel_estres`. Hay 28 nulos. Antes de imputar, comparen: ¿el nivel de estrés promedio difiere entre Juniors y Seniors?

Tarea: calculen la media de estrés por nivel, decidan si imputar con la media global o por grupo, y ejecuten la imputación.

```python
# Calculen media de nivel_estres por nivel
# Decidan la estrategia de imputación
# Ejecuten y verifiquen que no queden nulos
```

**Situación 3 — Reflexión sin código**

¿Qué columna del dataset les preocupa más en términos de calidad de datos y por qué? Escriban una respuesta de 2–3 líneas en una celda Markdown del notebook.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("healthcheck_colombia.csv")

df.head()

,id_trabajador,edad,ciudad,cargo,sector,nivel,anios_experiencia,horas_semana,salario_cop,nivel_estres,dias_ausencia_anio,satisfaccion_laboral,fecha_ingreso
0,1001,28,Bogotá,QA Engineer,Fintech,Junior,1,48.0,10859922.0,2.0,10.0,6.0,01/01/2018
1,1002,41,Barranquilla,UX Designer,Consultoría,Junior,9,56.0,6190583.0,8.0,6.0,8.0,04/01/2018
2,1003,50,Barranquilla,Desarrollador Junior,E-commerce,Lead,11,50.0,4099908.0,6.0,4.0,6.0,07/01/2018
3,1004,36,Bogotá,UX Designer,Healthtech,Senior,16,46.0,4664412.0,4.0,16.0,9.0,10/01/2018
4,1005,32,Medellín,Desarrollador Junior,Healthtech,Mid,7,53.0,7809792.0,5.0,19.0,9.0,13/01/2018


In [3]:
df.columns


Index(['id_trabajador', 'edad', 'ciudad', 'cargo', 'sector', 'nivel',
       'anios_experiencia', 'horas_semana', 'salario_cop', 'nivel_estres',
       'dias_ausencia_anio', 'satisfaccion_laboral', 'fecha_ingreso'],
      dtype='str')

In [4]:
df.isnull().sum()


id_trabajador            0
edad                     0
ciudad                  15
cargo                    0
sector                   0
nivel                    0
anios_experiencia        0
horas_semana             0
salario_cop             35
nivel_estres            28
dias_ausencia_anio       0
satisfaccion_laboral    22
fecha_ingreso            0
dtype: int64

In [5]:
columna = "dias_ausencia_anio"

Q1 = df[columna].quantile(0.25)
Q3 = df[columna].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df[(df[columna] < limite_inferior) | (df[columna] > limite_superior)]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Límite inferior:", limite_inferior)
print("Límite superior:", limite_superior)

outliers

Q1: 5.0
Q3: 14.0
IQR: 9.0
Límite inferior: -8.5
Límite superior: 27.5


,id_trabajador,edad,ciudad,cargo,sector,nivel,anios_experiencia,horas_semana,salario_cop,nivel_estres,dias_ausencia_anio,satisfaccion_laboral,fecha_ingreso
289,1290,42,Barranquilla,Product Manager,Consultoría,Mid,11,47.0,10101723.0,5.0,187.0,4.0,17/05/2020


### Decisión sobre el outlier

El valor de 187 días de ausencia anual fue detectado como outlier mediante el método IQR, porque supera el límite superior de 27.5 días calculado para la variable `dias_ausencia_anio`.

Aunque es un valor muy alto, no es imposible dentro de un año laboral aproximado de 240 días hábiles en Colombia. Por esta razón, se decide conservar la fila, pero dejar el dato identificado como caso atípico para análisis posteriores.

In [ ]:
# Situación 2: revisar valores nulos y medias de nivel_estres por nivel

print("Nulos en nivel_estres antes de imputar:")
print(df["nivel_estres"].isnull().sum())

print("\nMedia de nivel_estres por nivel:")
df.groupby("nivel")["nivel_estres"].mean()

Nulos en nivel_estres antes de imputar:
28

Media de nivel_estres por nivel:


nivel
Junior    5.099099
Lead      5.485714
Mid       5.273585
Senior    5.385714
Name: nivel_estres, dtype: float64

In [7]:
# Imputación de nivel_estres usando la media por grupo de nivel

df["nivel_estres"] = df.groupby("nivel")["nivel_estres"].transform(
    lambda x: x.fillna(x.mean())
)

print("Nulos en nivel_estres después de imputar:")
print(df["nivel_estres"].isnull().sum())

Nulos en nivel_estres después de imputar:
0


### Decisión de imputación para nivel_estres

Se decidió imputar los valores nulos de `nivel_estres` usando la media por grupo de la columna `nivel`, en lugar de usar una media global. Esta decisión es más adecuada porque el promedio de estrés cambia según el nivel del trabajador: Junior, Mid, Senior y Lead tienen medias diferentes.

De esta manera, cada valor faltante se reemplaza con una estimación más representativa del grupo al que pertenece el trabajador, evitando alterar demasiado el comportamiento real de los datos.

### Reflexión sobre calidad de datos

La columna que más preocupa en términos de calidad de datos es `nivel_estres`, porque tenía 28 valores nulos. Esta variable es importante para analizar el bienestar de los trabajadores, por lo que dejar esos datos faltantes podría generar conclusiones incompletas o sesgadas.

También se debe tener cuidado con `dias_ausencia_anio`, porque aparece un valor atípico de 187 días. Aunque no es imposible dentro de un año laboral, sí debe quedar identificado para evitar interpretaciones equivocadas en análisis posteriores.